# **Preparing Indices**
*Notebook created Dec. 17, 2025*

This notebook documents the processing of ISO index files in preparation for evaluating the relationship between SWM and MJO/BSISO. All processes will be performed locally.

## **1. Downloading Datasets**
The following datasets will be used in investigating MJO and BSISO.
| Source | MJO | BSISO | Period | Temporal Resolution | File Extension |
| :--- | :---: | :---: | :---: | :---: | :---: |
| [Bimodal ISO Index](iprc.soest.hawaii.edu/users/kazuyosh/Bimodal_ISO.html) | <span style="color:green">**Yes**</span> | <span style="color:green">**Yes**</span> | 1979 – 2020 | Daily | .txt |

## **2. Preparing the Datasets**
**Given:** The historical PCs of the bimodal ISO index is archived in .txt files.

**Objective:** Generate .csv files from the raw .txt files.

## **3. Filtering Active ISO Days**
**Given:** The historical bimodal ISO index provides daily normalized amplitudes for MJO/BSISO.

**Objective:** Generate new .csv files containing only active days (`nrm > 1`) within June to September.

In [1]:
# MJO filter
import pandas as pd

input_file = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\MJO.csv"
output_file = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\MJO_filtered.csv"

df = pd.read_csv(input_file)

df["Amp(nrm)"] = pd.to_numeric(df["Amp(nrm)"], errors="coerce")
df_filtered = df[
    (df["Amp(nrm)"] > 1) &
    (df["mon"].isin([5, 6, 7, 8, 9]))
]

df_filtered.to_csv(output_file, index=False)

print(f"Filtered file saved to:\n{output_file}")

Filtered file saved to:
C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\MJO_filtered.csv


In [2]:
import pandas as pd
import numpy as np

input_file = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\MJO_filtered.csv"
output_file = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\MJO_filtered_stats.csv"

df = pd.read_csv(input_file)
df["Amp(nrm)"] = pd.to_numeric(df["Amp(nrm)"], errors="coerce")
f_filtered = df[
    (df["Amp(nrm)"] > 1) &
    (df["mon"].isin([5, 6, 7, 8, 9]))
].copy()

def get_pentad_id(row):
    day = row['day']
    month = row['mon']
    
    if day >= 26:
        pentad_in_month = 6
    else:
        pentad_in_month = (day - 1) // 5 + 1
        
    return (month - 1) * 6 + pentad_in_month

df_filtered['pentad_id'] = df_filtered.apply(get_pentad_id, axis=1)
days_count = df_filtered.groupby('phase').size().reset_index(name='total_days')
pentad_counts = df_filtered.groupby(['year', 'pentad_id', 'phase']).size().reset_index(name='days_active')
full_pentads = pentad_counts[pentad_counts['days_active'] >= 5].groupby('phase').size().reset_index(name='full_pentads_count')
majority_pentads = pentad_counts[pentad_counts['days_active'] >= 3].groupby('phase').size().reset_index(name='majority_pentads_count')

summary = days_count.merge(full_pentads, on='phase', how='left')
summary = summary.merge(majority_pentads, on='phase', how='left')
summary = summary.fillna(0)
summary['full_pentads_count'] = summary['full_pentads_count'].astype(int)
summary['majority_pentads_count'] = summary['majority_pentads_count'].astype(int)

print("Summary Statistics by Phase:")
print(summary)

Summary Statistics by Phase:
   phase  total_days  full_pentads_count  majority_pentads_count
0      1          70                   1                      12
1      2          45                   1                       9
2      3          92                   3                      19
3      4          88                   5                      15
4      5          42                   1                       5
5      6          49                   0                       8
6      7         123                   6                      25
7      8         118                   5                      24


In [3]:
import pandas as pd

input_file = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\MJO_filtered.csv"
output_full = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\MJO_filtered_FullPentads.csv"
output_majority = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\MJO_filtered_MajorityPentads.csv"

df = pd.read_csv(input_file)
df["Amp(nrm)"] = pd.to_numeric(df["Amp(nrm)"], errors="coerce")

def get_pentad_id(row):
    day = row['day']
    month = row['mon']
    
    if day >= 26:
        pentad_in_month = 6
    else:
        pentad_in_month = (day - 1) // 5 + 1
        
    return (month - 1) * 6 + pentad_in_month

df['pentad_id'] = df.apply(get_pentad_id, axis=1)
df['days_active_in_pentad'] = df.groupby(['year', 'pentad_id', 'phase'])['day'].transform('count')
df_full = df[df['days_active_in_pentad'] >= 5].copy()
df_full_clean = df_full.drop(columns=['pentad_id', 'days_active_in_pentad'])
df_full_clean.to_csv(output_full, index=False)
print(f"Full Pentads file saved to:\n{output_full}")

df_majority = df[df['days_active_in_pentad'] >= 3].copy()
df_majority_clean = df_majority.drop(columns=['pentad_id', 'days_active_in_pentad'])
df_majority_clean.to_csv(output_majority, index=False)
print(f"Majority Pentads file saved to:\n{output_majority}")

Full Pentads file saved to:
C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\MJO_filtered_FullPentads.csv
Majority Pentads file saved to:
C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\MJO_filtered_MajorityPentads.csv


In [4]:
# BSISO filter
import pandas as pd

input_file = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\BSISO.csv"
output_file = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\BSISO_filtered.csv"

df = pd.read_csv(input_file)

df["Amp(nrm)"] = pd.to_numeric(df["Amp(nrm)"], errors="coerce")
df_filtered = df[
    (df["Amp(nrm)"] > 1) &
    (df["mon"].isin([5, 6, 7, 8, 9]))
]

df_filtered.to_csv(output_file, index=False)

print(f"Filtered file saved to:\n{output_file}")

Filtered file saved to:
C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\BSISO_filtered.csv


In [5]:
import pandas as pd
import numpy as np

input_file = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\BSISO_filtered.csv"
output_file = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\BSISO_filtered_stats.csv"

df = pd.read_csv(input_file)
df["Amp(nrm)"] = pd.to_numeric(df["Amp(nrm)"], errors="coerce")
f_filtered = df[
    (df["Amp(nrm)"] > 1) &
    (df["mon"].isin([5, 6, 7, 8, 9]))
].copy()

def get_pentad_id(row):
    day = row['day']
    month = row['mon']
    
    if day >= 26:
        pentad_in_month = 6
    else:
        pentad_in_month = (day - 1) // 5 + 1
        
    return (month - 1) * 6 + pentad_in_month

df_filtered['pentad_id'] = df_filtered.apply(get_pentad_id, axis=1)
days_count = df_filtered.groupby('phase').size().reset_index(name='total_days')
pentad_counts = df_filtered.groupby(['year', 'pentad_id', 'phase']).size().reset_index(name='days_active')
full_pentads = pentad_counts[pentad_counts['days_active'] >= 5].groupby('phase').size().reset_index(name='full_pentads_count')
majority_pentads = pentad_counts[pentad_counts['days_active'] >= 3].groupby('phase').size().reset_index(name='majority_pentads_count')

summary = days_count.merge(full_pentads, on='phase', how='left')
summary = summary.merge(majority_pentads, on='phase', how='left')
summary = summary.fillna(0)
summary['full_pentads_count'] = summary['full_pentads_count'].astype(int)
summary['majority_pentads_count'] = summary['majority_pentads_count'].astype(int)

print("Summary Statistics by Phase:")
print(summary)

Summary Statistics by Phase:
   phase  total_days  full_pentads_count  majority_pentads_count
0      1         508                  24                      97
1      2         492                  23                      93
2      3         507                  24                      98
3      4         521                  28                     107
4      5         500                  28                      97
5      6         462                  23                      86
6      7         448                  18                      92
7      8         521                  34                     100


In [6]:
import pandas as pd

input_file = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\BSISO_filtered.csv"
output_full = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\BSISO_filtered_FullPentads.csv"
output_majority = r"C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\BSISO_filtered_MajorityPentads.csv"

df = pd.read_csv(input_file)
df["Amp(nrm)"] = pd.to_numeric(df["Amp(nrm)"], errors="coerce")

def get_pentad_id(row):
    day = row['day']
    month = row['mon']
    
    if day >= 26:
        pentad_in_month = 6
    else:
        pentad_in_month = (day - 1) // 5 + 1
        
    return (month - 1) * 6 + pentad_in_month

df['pentad_id'] = df.apply(get_pentad_id, axis=1)
df['days_active_in_pentad'] = df.groupby(['year', 'pentad_id', 'phase'])['day'].transform('count')
df_full = df[df['days_active_in_pentad'] >= 5].copy()
df_full_clean = df_full.drop(columns=['pentad_id', 'days_active_in_pentad'])
df_full_clean.to_csv(output_full, index=False)
print(f"Full Pentads file saved to:\n{output_full}")

df_majority = df[df['days_active_in_pentad'] >= 3].copy()
df_majority_clean = df_majority.drop(columns=['pentad_id', 'days_active_in_pentad'])
df_majority_clean.to_csv(output_majority, index=False)
print(f"Majority Pentads file saved to:\n{output_majority}")

Full Pentads file saved to:
C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\BSISO_filtered_FullPentads.csv
Majority Pentads file saved to:
C:\Users\Nitro 5\Documents\MS\Thesis\GitHub\MS_Thesis_SWM\01_data\02_processed\BSISO_filtered_MajorityPentads.csv


### **A. Collapsing Active ISO Days to Pentads**
